In [ ]:
from langchain.tools import tool
from langgraph.cache import base
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
import sqlite3
from typing import List, Any, Dict
import os

load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL")
tavily_key =  os.getenv("TAVILY_API_KEY")

search_tool = TavilySearch(
    max_results=5,
    topic = "general"
)
# 数据库查询工具
class DatabaseQueryClient:
    def __init__(self, db_path: str = "agent_data.db"):
        self.db_path = db_path

    def select(
        self,
        table: str,
        columns: str = "*",
        where: str = "",
        params: List[Any] = None,
        limit: int = 100
    ) -> Dict[str, Any]:
        params = params or []
        sql_parts = [f"SELECT {columns} FROM {table}"]
        if where.strip():
            sql_parts.append(f"WHERE {where}")
        sql_parts.append(f"LIMIT {limit}")
        sql = " ".join(sql_parts)

        try:
            conn = sqlite3.connect(self.db_path)
            conn.row_factory = sqlite3.Row
            cur = conn.cursor()
            cur.execute(sql, tuple(params))
            rows = [dict(r) for r in cur.fetchall()]
            conn.close()
            return {"success": True, "count": len(rows), "data": rows}
        except Exception as e:
            return {"success": False, "error": str(e), "data": []}

# 数据库查询参数
class DBQuerySchema(BaseModel):
    table: str = Field(description="要查询的数据库名")
    columns: str = Field(description="要查询的列名，用逗号分隔")
    where: str = Field(description="查询条件，用SQL格式")
    params: List[Any] = Field(description="查询参数，用于替换SQL中的占位符")
    limit: int = Field(default=50, description="结果上限")

db_client = DatabaseQueryClient("agent_data.db")

@tool(args_schema=DBQuerySchema)
def query_db( table: str,
    columns: str,
    where: str,
    params: List[Any],
    limit: int = 50) -> Dict[str, Any]:
    """
    查询本地SQLite数据库 agent_data.db
    注意：构造where条件时，值使用 ? 占位符，真实数值放入params数组，禁止直接拼接字符串！
    """
    return db_client.select(
        table=table,
        columns=columns,
        where=where,
        params=params,
        limit=limit
    )

model = ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url=base_url,
    extra_body={"enable_thinking": False},
)
system_prompt="""
你是一个数据库查询助手查询本地数据库 agent_data.db。
"""
agent = create_agent(
    model=model,
    tools=[query_db],
    system_prompt=system_prompt,
)


result = agent.invoke(
    {"messages": [{"role": "user", "content": "查询下agent_data.db数据库里小明的数据"}]}
)
print(result["messages"][-1].content)


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver  
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.tools import tool
from typing import List, Dict, Any
from datetime import datetime

import os
import json
import sqlite3


class DatebaseQueryClient:
    def __init__(self, db_path: str = "gold_data.db"):
        self.db_path = db_path

    def select(self,table:str,columns: str = "*",where: str = "",params: List[Any] = None,limit : int = 100) -> Dict[str,Any]:
        params = params or []
        sql_parts = [f"SELECT {columns} FROM {table}"]
        if where.strip():
           sql_parts.append(f"WHERE {where}")
        sql_parts.append(f"LIMIT {limit}")
        sql = " ".join(sql_parts)

        try:
            conn = sqlite3.connect(self.db_path)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()
            cursor.execute(sql,tuple(params))
            rows = [dict(r) for r in cursor.fetchall()]
            conn.close()
            return {"success":True,"count":len(rows),"data":rows}
        except sqlite3.Error as e:
            return {"success":False,"error":str(e),"data":[]}

db_client = DatebaseQueryClient()


class GoldDataSchema(BaseModel):
    table: str = Field(description="黄金数据表名")
    columns: str = Field(description="查询列，默认所有列")
    where: str = Field(description="查询条件，用SQL格式")
    params: List[Any] = Field(description="查询参数，替换SQL中的占位符")
    limit: int = Field(default=50,description="查询限制，默认50条")

@tool(args_schema=GoldDataSchema)
def query_db(table:str,columns:str,where:str,params:List[Any],limit:int=50) -> Dict[str,Any]:
    """
    查询本地SQLite数据库 gold_data.db,查看每个人的黄金数据。
    注意：构造where条件时，值使用 ? 占位符，真实数值放入params数组，禁止直接拼接字符串！
    """
    return db_client.select(
        table=table,
        columns=columns,
        where=where,
        params=params,
        limit=limit
    )

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)
@tool
def get_day() -> str:
    """
    获取当天日期
    """
    now = datetime.now()
    day_date = now.strftime("%Y-%m-%d")
    return f"今天是{day_date}"

model = ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url=base_url,
    extra_body={"enable_thinking": False},
)

SYSTEM_PROMPT = """
你是个人黄金资产核算助手，严格遵循执行流程：
1. 用户询问黄金价值时，**第一步调用 query_db 获取对应人的黄金数据**
2  把黄金数据中的数量转换为克重
3. 调用 get_day 获取当前日期
4. 调用 tavily_search_tool 查询get_day()的金价。
5. 从工具返回内容中提取数字：总克重、金价（元/克）
6. 计算公式：总价值 = 黄金克重 × 实时金价

【金价筛选强制规则】
1. 国际美元金价不可直接使用；
2. 周大福、周生生等金店首饰挂牌零售价，不用于估算投资金条市值；
3. 优先选取：国内基础金价、AU9999原料金价、黄金回收参考价；
4. 获取到的金价必须和get_day()的日期一致；
5. 记录最终选用的金价，并在回答中标明价格类型。
6. 计算公式：总价值 = 黄金总克重 × 选取的人民币金价（元/克）
7. 最终回答清晰列出：人名、各笔黄金重量、换算后总克重、选用金价、金价类型、估算总金额

规则：
- 禁止编造克重和金价，必须通过工具获取；
- 如果搜索出来多个金价，优先选用足金零售价，并在答案中说明；
- 金额保留2位小数。
"""
tools = [query_db,tavily_search_tool,get_day]
checkpointer = InMemorySaver()


agent =  create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "gold_calc_001"}}
    user_question = "小红手里的黄金现在大概价值多少钱？"

    response = agent.invoke(
        {"messages": [("user", user_question)]},
        config=config
    )

    # 打印完整执行链路
    for msg in response["messages"]:
        print(f"\n【{msg.type}】")
        print(msg.content)
